# Sparsity Thresholds With Row-Wise Features

Compare several sparse-feature removal thresholds together with a minimal set of row-wise aggregate features using the current best `LGBM` parameters.

In [1]:
import sys

sys.path.append("../")

import json
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, root_mean_squared_log_error
from sklearn.model_selection import KFold, train_test_split

from src.features import add_rowwise_features, get_sparse_columns_to_keep
from src.loader import Loader
from src.modeling import build_lgbm_regressor

In [3]:
SEED = 42
TEST_SIZE = 0.33
CV = 5
THRESHOLDS = [None, 0.975, 0.98, 0.9825, 0.985, 0.9875, 0.99]

In [4]:
loader = Loader()
df = loader.load("../data/processed_data.csv")
df.shape

(4459, 4732)

In [5]:
X = df.drop(columns="target")
y = df["target"]
y_log = np.log1p(y)

(X.shape, y.shape)

((4459, 4731), (4459,))

Sparse filtering is applied only on the original features. Row-wise features are added after filtering so that each threshold is evaluated on its own final feature space.

In [8]:
default_params = {
    "n_estimators": 500,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": 8,
    "min_child_samples": 20,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.05,
    "reg_lambda": 0.05,
    "min_split_gain": 0.0,
}

summary_path = Path("../artifacts/optuna_lgbm/optuna_summary.json")
if summary_path.exists():
    with open(summary_path) as f:
        best_params = json.load(f)["best_params"]
else:
    best_params = default_params

best_params

{'n_estimators': 439,
 'learning_rate': 0.019291857433628438,
 'num_leaves': 32,
 'max_depth': 11,
 'min_child_samples': 5,
 'subsample': 0.9872554086372487,
 'subsample_freq': 5,
 'colsample_bytree': 0.7408418143236152,
 'reg_alpha': 0.025576812693216197,
 'reg_lambda': 0.718015922751547,
 'min_split_gain': 0.04545640466117268}

In [10]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw, y_train_log, y_test_log = train_test_split(
    X,
    y,
    y_log,
    test_size=TEST_SIZE,
    random_state=SEED,
)

cv = KFold(n_splits=CV, shuffle=True, random_state=SEED)

In [11]:
def evaluate_threshold(zero_share_threshold: float | None) -> dict:
    cv_scores = []
    cv_feature_counts = []

    for fold_train_idx, fold_valid_idx in cv.split(X_train_raw, y_train_log):
        X_fold_train_raw = X_train_raw.iloc[fold_train_idx]
        X_fold_valid_raw = X_train_raw.iloc[fold_valid_idx]
        y_fold_train_log = y_train_log.iloc[fold_train_idx]
        y_fold_valid_log = y_train_log.iloc[fold_valid_idx]

        keep_columns = get_sparse_columns_to_keep(X_fold_train_raw, zero_share_threshold)
        X_fold_train = add_rowwise_features(X_fold_train_raw[keep_columns])
        X_fold_valid = add_rowwise_features(X_fold_valid_raw[keep_columns])

        cv_feature_counts.append(X_fold_train.shape[1])

        model = build_lgbm_regressor(best_params)
        model.fit(X_fold_train, y_fold_train_log)
        y_fold_pred_log = model.predict(X_fold_valid)
        fold_rmsle = root_mean_squared_error(y_fold_valid_log, y_fold_pred_log)
        cv_scores.append(fold_rmsle)

    keep_columns = get_sparse_columns_to_keep(X_train_raw, zero_share_threshold)
    X_train_final = add_rowwise_features(X_train_raw[keep_columns])
    X_test_final = add_rowwise_features(X_test_raw[keep_columns])

    model = build_lgbm_regressor(best_params)
    model.fit(X_train_final, y_train_log)
    y_pred_log = model.predict(X_test_final)
    y_pred = np.expm1(y_pred_log)
    y_pred = np.clip(y_pred, 0, None)

    return {
        "threshold": "none" if zero_share_threshold is None else zero_share_threshold,
        "base_features_kept": len(keep_columns),
        "final_features": X_train_final.shape[1],
        "cv_final_features_mean": float(np.mean(cv_feature_counts)),
        "cv_rmsle_mean": float(np.mean(cv_scores)),
        "cv_rmsle_std": float(np.std(cv_scores)),
        "holdout_rmsle": float(root_mean_squared_log_error(y_test_raw, y_pred)),
        "holdout_rmse": float(root_mean_squared_error(y_test_raw, y_pred)),
        "holdout_mae": float(mean_absolute_error(y_test_raw, y_pred)),
        "holdout_r2": float(r2_score(y_test_raw, y_pred)),
    }

In [12]:
results = [evaluate_threshold(threshold) for threshold in THRESHOLDS]
results_df = pd.DataFrame(results).sort_values(by="cv_rmsle_mean").reset_index(drop=True)
results_df.style.format({
    "cv_rmsle_mean": "{:,.4f}",
    "cv_rmsle_std": "{:,.4f}",
    "holdout_rmsle": "{:,.4f}",
    "holdout_rmse": "{:,.0f}",
    "holdout_mae": "{:,.0f}",
    "holdout_r2": "{:,.4f}",
    "cv_final_features_mean": "{:,.1f}",
})

,threshold,base_features_kept,final_features,cv_final_features_mean,cv_rmsle_mean,cv_rmsle_std,holdout_rmsle,holdout_rmse,holdout_mae,holdout_r2
0,0.987500,2482,2490,"2,506.2",1.3644,0.0302,1.3908,"6,942,105","3,895,660",0.2449
1,0.985000,2415,2423,"2,415.8",1.3684,0.0368,1.3807,"6,903,628","3,874,841",0.2533
2,0.990000,2669,2677,"2,690.8",1.3705,0.0323,1.3824,"6,940,756","3,898,643",0.2452
3,0.975000,1832,1840,"1,843.2",1.3726,0.0348,1.3808,"6,883,340","3,855,308",0.2576
4,0.982500,2287,2295,"2,301.2",1.3736,0.0320,1.3859,"6,930,249","3,890,769",0.2475
5,0.980000,2113,2121,"2,133.6",1.3751,0.0342,1.3801,"6,935,724","3,869,397",0.2463
6,none,4731,4739,"4,739.0",1.3819,0.0405,1.3931,"6,977,905","3,919,146",0.2371


In [13]:
ARTIFACTS_DIR = Path("../artifacts/sparsity_thresholds_rowwise_lgbm")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(ARTIFACTS_DIR / "sparsity_threshold_results.csv", index=False)

summary = {
    "target_transform": "log1p",
    "primary_metric": "rmsle",
    "thresholds": ["none" if threshold is None else threshold for threshold in THRESHOLDS],
    "model_params": best_params,
    "rowwise_features": [
        "non_zero_count",
        "non_zero_ratio",
        "row_sum",
        "row_mean",
        "row_std",
        "row_max",
        "nz_mean",
        "nz_std",
    ],
    "results": results_df.to_dict(orient="records"),
}

with open(ARTIFACTS_DIR / "sparsity_threshold_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

## How To Read The Results

- Read the ranking by `cv_rmsle_mean` first.
- Accept a threshold only if the holdout `RMSLE` does not regress materially.
- If one threshold wins clearly, rerun `Optuna` on that filtered feature space rather than trying many more sparse cutoffs.